# M1 Notebook 09 — Derivatives, Gradients, Jacobians, and Hessians

**Notebook ID:** M1_N09  
**Status:** Runnable first edition  
**Random seed:** 42

> Derivatives measure local change. Gradients collect directional sensitivities. Jacobians describe vector transformations. Hessians describe local curvature.


## 1. Learning objectives

1. Interpret first and second derivatives.
2. Compare forward, backward, and central differences.
3. Compute gradients and directional derivatives.
4. Construct Jacobian matrices.
5. Construct Hessian matrices.
6. Use first-order Taylor approximations.
7. Connect derivatives to optimization, AI, and Decision Intelligence.


In [ ]:
from srai_math.utils import environment_info, set_seed
from srai_math.calculus import (
    derivative,
    directional_derivative,
    gradient,
    hessian,
    jacobian,
    second_derivative,
    taylor_first_order,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
set_seed(42)
environment_info()


## 2. Scalar derivative

\[
f'(x)
=
\lim_{h\to0}
\frac{f(x+h)-f(x)}{h}.
\]

The derivative is the local rate of change and the slope of the tangent line.


In [ ]:
f = lambda x: x**3
x0 = 2.0

estimates = {
    "forward": derivative(f, x0, method="forward"),
    "backward": derivative(f, x0, method="backward"),
    "central": derivative(f, x0, method="central"),
    "exact": 3*x0**2,
}
estimates


In [ ]:
assert np.isclose(estimates["central"], estimates["exact"], rtol=1e-6)
print("Scalar derivative verified.")


## 3. Step-size experiment

In [ ]:
step_sizes = 10.0 ** (-np.arange(1, 13))
errors = []

for h in step_sizes:
    estimate = derivative(f, x0, h=h, method="central")
    errors.append(abs(estimate - 12.0))

step_table = pd.DataFrame({
    "h": step_sizes,
    "absolute_error": errors,
})
step_table


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(step_sizes, errors)
ax.set_xlabel("Step size h")
ax.set_ylabel("Absolute error")
ax.set_title("Finite-Difference Error versus Step Size")
plt.show()


Very small \(h\) does not guarantee better results because floating-point cancellation eventually dominates.


## 4. Second derivative

\[
f''(x)
=
\lim_{h\to0}
\frac{f(x+h)-2f(x)+f(x-h)}{h^2}.
\]

The second derivative measures local curvature.


In [ ]:
second = second_derivative(f, 2.0)
assert np.isclose(second, 12.0, rtol=1e-4)
second


## 5. Gradient

For a scalar-valued function

\[
f:\mathbb R^n\to\mathbb R,
\]

the gradient is

\[
\nabla f(\mathbf x)
=
\begin{bmatrix}
\partial f/\partial x_1\\
\vdots\\
\partial f/\partial x_n
\end{bmatrix}.
\]


In [ ]:
surface = lambda x: x[0]**2 + 3*x[1]**2
point = np.array([2.0, 1.0])

g = gradient(surface, point)
assert np.allclose(g, [4.0, 6.0], atol=1e-6)
g


## 6. Gradient visualization

In [ ]:
x1 = np.linspace(-2, 2, 120)
x2 = np.linspace(-2, 2, 120)
X1, X2 = np.meshgrid(x1, x2)
Z = X1**2 + 3*X2**2

fig, ax = plt.subplots(figsize=(6, 5))
contours = ax.contour(X1, X2, Z, levels=15)
ax.clabel(contours, inline=True, fontsize=8)
ax.quiver(point[0], point[1], g[0], g[1], angles="xy", scale_units="xy", scale=1)
ax.scatter([point[0]], [point[1]])
ax.set_xlabel("x₁")
ax.set_ylabel("x₂")
ax.set_title("Gradient on a Contour Map")
plt.show()


The gradient points in the direction of steepest local increase. The negative gradient points toward steepest local decrease.


## 7. Directional derivative

For a unit direction \(\mathbf u\),

\[
D_{\mathbf u}f(\mathbf x)
=
\nabla f(\mathbf x)^\top\mathbf u.
\]


In [ ]:
direction = np.array([1.0, 1.0])
directional = directional_derivative(surface, point, direction)
analytic = g @ (direction / np.linalg.norm(direction))

assert np.isclose(directional, analytic, rtol=1e-6)
directional, analytic


## 8. Jacobian

For

\[
F:\mathbb R^n\to\mathbb R^m,
\]

the Jacobian is

\[
J_F(\mathbf x)
=
\left[
\frac{\partial F_i}{\partial x_j}
\right].
\]


In [ ]:
vector_function = lambda x: np.array([
    x[0] + x[1],
    x[0] * x[1],
])

J = jacobian(vector_function, [2.0, 3.0])
expected_J = np.array([
    [1.0, 1.0],
    [3.0, 2.0],
])

assert np.allclose(J, expected_J, atol=1e-6)
J


## 9. Hessian

For a scalar-valued multivariate function,

\[
H_f(\mathbf x)
=
\left[
\frac{\partial^2 f}
{\partial x_i\partial x_j}
\right].
\]

The Hessian describes local curvature.


In [ ]:
quadratic = lambda x: (
    x[0]**2
    + 3*x[1]**2
    + 2*x[0]*x[1]
)

H = hessian(quadratic, [1.0, 2.0])
expected_H = np.array([
    [2.0, 2.0],
    [2.0, 6.0],
])

assert np.allclose(H, expected_H, atol=1e-4)
H


## 10. Curvature classification

In [ ]:
eigenvalues = np.linalg.eigvalsh(H)

classification = (
    "positive definite"
    if np.all(eigenvalues > 0)
    else "negative definite"
    if np.all(eigenvalues < 0)
    else "indefinite"
)

{
    "Hessian_eigenvalues": eigenvalues,
    "classification": classification,
}


A positive-definite Hessian indicates local convex curvature. An indefinite Hessian indicates a saddle-like region.


## 11. First-order Taylor approximation

Near \(\mathbf x_0\),

\[
f(\mathbf x)
\approx
f(\mathbf x_0)
+
\nabla f(\mathbf x_0)^\top
(\mathbf x-\mathbf x_0).
\]


In [ ]:
x_center = np.array([1.0, 1.0])
x_target = np.array([1.05, 0.95])

f_center = surface(x_center)
g_center = gradient(surface, x_center)

approximation = taylor_first_order(
    f_center,
    g_center,
    x_target,
    x_center,
)
actual = surface(x_target)

{
    "first_order_approximation": approximation,
    "actual_value": actual,
    "absolute_error": abs(actual - approximation),
}


## 12. Statistics interpretation

Derivatives support:

- likelihood optimization;
- score functions;
- Fisher information;
- sensitivity analysis;
- delta-method approximations;
- nonlinear regression.


## 13. AI interpretation

Gradients and Jacobians are central to:

- backpropagation;
- gradient descent;
- automatic differentiation;
- adversarial sensitivity;
- saliency;
- neural-network optimization.

Hessians support curvature analysis, second-order optimization, and uncertainty approximations.


## 14. Decision Intelligence case — Policy sensitivity

Consider a simplified welfare function of health and education investment.


In [ ]:
def welfare(x):
    health, education = x
    return (
        4*np.sqrt(health)
        + 5*np.sqrt(education)
        - 0.05*(health + education)**2
    )

policy = np.array([20.0, 20.0])
policy_gradient = gradient(welfare, policy)
policy_hessian = hessian(welfare, policy)

pd.Series(
    policy_gradient,
    index=["Health marginal effect", "Education marginal effect"],
)


In [ ]:
pd.DataFrame(
    policy_hessian,
    index=["Health", "Education"],
    columns=["Health", "Education"],
)


### Interpretation

The gradient estimates local marginal changes in modeled welfare. The Hessian estimates how those marginal effects change and interact. Neither object determines policy by itself: constraints, uncertainty, distributional effects, ethics, and implementation capacity must also be considered.


## 15. Engineering notes

- Finite differences are sensitive to step size.
- Central differences are usually more accurate than one-sided differences.
- High-dimensional Hessians are expensive to compute and store.
- Automatic differentiation is often preferable for large models.
- Numerical derivatives can be corrupted by noise and discontinuities.
- Derivatives describe local behavior, not global outcomes.


## 16. Common errors

- Confusing derivative values with function values.
- Forgetting to normalize directional vectors.
- Mixing Jacobian conventions.
- Assuming a positive diagonal implies a positive-definite Hessian.
- Using finite differences across discontinuities.
- Treating local sensitivities as causal effects.


## 17. Exercises

### Level A
Explain gradients, Jacobians, and Hessians.

### Level B
Derive the gradient and Hessian of a quadratic form.

### Level C
Implement numerical gradient checking.

### Capstone
Construct a multi-sector policy objective, compute gradients and Hessians, evaluate local sensitivities, and document why the result is not sufficient for automatic policy selection.


## 18. Key insight

Derivatives convert change into mathematics. Gradients reveal local sensitivity, Jacobians describe local transformations, and Hessians reveal curvature. These objects form the computational foundation of optimization, machine learning, and Decision Intelligence.
